# Week 6: Model Evaluation

**Where we are.** Last week we trained our first model. This week we focus on **evaluation**: confusion matrix, accuracy, precision, recall, F1, overfitting, and tuning.

**How this notebook works (same as last week).** For each idea, a **demo** on a sample dataset, then **Your Turn** to repeat it on your own dataset.

**One change from last week:** everyone now uses the same dataset in the Your Turn sections (loaded for you below), so we can compare results as a class.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, ConfusionMatrixDisplay,
                             classification_report)

pd.set_option('display.max_columns', 50)
pd.set_option('display.precision', 3)
print('Libraries loaded.')

---
## Step 0 — Recap: Load, Split, Train

**Demo dataset:** breast cancer data (same as last week). Note: in scikit-learn's version, 0 = malignant and 1 = benign. We flip it so 1 = malignant, so "positive" means the case we're trying to catch.

In [ ]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer(as_frame=True)
df_demo = data.frame.copy()
df_demo['target'] = 1 - df_demo['target']   # 1 = malignant now

X_demo = df_demo.drop(columns=['target'])
y_demo = df_demo['target']

X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(
    X_demo, y_demo, test_size=0.2, random_state=42, stratify=y_demo
)

model_demo = LogisticRegression(max_iter=5000)
model_demo.fit(X_train_d, y_train_d)
preds_demo = model_demo.predict(X_test_d)

print('Shape:', df_demo.shape, ' Train/Test:', X_train_d.shape, X_test_d.shape)

### Your Turn — Load, split, train on your dataset

**Your dataset: Pima Indians Diabetes.** Target `Outcome`: 1 = diabetes, 0 = no diabetes. A community clinic wants to flag patients for a follow-up test, so 1 (diabetes) is the "positive" we want to catch.

Some columns (like `Insulin`) use 0 to mean "not measured" — we're leaving that as-is to keep focus on evaluation.

In [ ]:
url = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv'
columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness',
           'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome']
df_yours = pd.read_csv(url, header=None, names=columns)

X_yours = df_yours.drop(columns=['Outcome'])
y_yours = df_yours['Outcome']

X_train_y, X_test_y, y_train_y, y_test_y = train_test_split(
    X_yours, y_yours, test_size=0.2, random_state=42, stratify=y_yours
)

model_yours = LogisticRegression(max_iter=5000)
model_yours.fit(X_train_y, y_train_y)
preds_yours = model_yours.predict(X_test_y)

print('Shape:', df_yours.shape, ' Train/Test:', X_train_y.shape, X_test_y.shape)

---
## Step 1 — Confusion Matrix, Accuracy, and the Accuracy Trap

The confusion matrix counts predictions by type:

|  | Predicted Negative | Predicted Positive |
|---|---|---|
| **Actually Negative** | True Negative (TN) | False Positive (FP) — false alarm |
| **Actually Positive** | False Negative (FN) — missed case | True Positive (TP) |

**Accuracy** = (TN + TP) / everything. It can mislead on imbalanced data: a model that always predicts the majority class can score high accuracy while catching nothing.

In [ ]:
cm = confusion_matrix(y_test_d, preds_demo)
tn, fp, fn, tp = cm.ravel()
print(f'TN: {tn}  FP: {fp}  FN: {fn} (missed)  TP: {tp}')

ConfusionMatrixDisplay.from_predictions(
    y_test_d, preds_demo, display_labels=['Benign', 'Malignant'], cmap='Blues')
plt.title('Confusion Matrix — Demo Model')
plt.show()

# Compare to a baseline that does no learning at all
dummy_d = DummyClassifier(strategy='most_frequent').fit(X_train_d, y_train_d)
print('Accuracy, do-nothing baseline:', accuracy_score(y_test_d, dummy_d.predict(X_test_d)))
print('Accuracy, logistic regression:', accuracy_score(y_test_d, preds_demo))

### Your Turn — Confusion matrix and accuracy check

In [ ]:
cm_y = confusion_matrix(y_test_y, preds_yours)
tn_y, fp_y, fn_y, tp_y = cm_y.ravel()
print(f'TN: {tn_y}  FP: {fp_y}  FN: {fn_y} (missed)  TP: {tp_y}')

ConfusionMatrixDisplay.from_predictions(
    y_test_y, preds_yours, display_labels=['No diabetes', 'Diabetes'], cmap='Blues')
plt.title('Confusion Matrix — Your Model')
plt.show()

dummy_y = DummyClassifier(strategy='most_frequent').fit(X_train_y, y_train_y)
print('Accuracy, do-nothing baseline:', accuracy_score(y_test_y, dummy_y.predict(X_test_y)))
print('Accuracy, logistic regression:', accuracy_score(y_test_y, preds_yours))

**Your answers:**

1. In plain English: of the real diabetes cases, how many did your model catch and how many did it miss? Of the people flagged, how many were false alarms?
2. How much better is your model's accuracy than the do-nothing baseline? For a clinic, what does a missed case cost versus a false alarm — which is worse?

---
## Step 2 — Precision, Recall, F1

| Metric | Formula | Question it answers |
|---|---|---|
| Precision | TP / (TP + FP) | When the model raises an alarm, how often is it right? |
| Recall | TP / (TP + FN) | Of all the real cases, how many did the model catch? |
| F1 | 2PR / (P + R) | One number balancing both. |

High recall matters when missing a case is costly (screening, fraud). High precision matters when false alarms are costly (spam filters, wasted review time).

In [ ]:
print(f'Accuracy:  {accuracy_score(y_test_d, preds_demo):.3f}')
print(f'Precision: {precision_score(y_test_d, preds_demo):.3f}')
print(f'Recall:    {recall_score(y_test_d, preds_demo):.3f}')
print(f'F1:        {f1_score(y_test_d, preds_demo):.3f}')
print()
print(classification_report(y_test_d, preds_demo, target_names=['Benign', 'Malignant']))

### Your Turn

In [ ]:
print(f'Accuracy:  {accuracy_score(y_test_y, preds_yours):.3f}')
print(f'Precision: {precision_score(y_test_y, preds_yours):.3f}')
print(f'Recall:    {recall_score(y_test_y, preds_yours):.3f}')
print(f'F1:        {f1_score(y_test_y, preds_yours):.3f}')
print()
print(classification_report(y_test_y, preds_yours, target_names=['No diabetes', 'Diabetes']))

**Your answers:**

1. In your own words, what do your precision and recall mean for a clinic patient?
2. For diabetes screening, would you rather have higher precision or higher recall? Why?

---
## Step 3 — Thresholds

Classifiers output a probability, and `.predict()` applies a default threshold of 0.5. Lowering the threshold flags more cases (recall up, precision down); raising it flags fewer (precision up, recall down). Choosing the threshold is a business decision about which mistake you can better afford.

In [ ]:
def threshold_table(y_true, probs, thresholds=np.arange(0.1, 1.0, 0.1)):
    y_true = np.asarray(y_true)
    rows = []
    for t in thresholds:
        pred = (probs >= t).astype(int)
        rows.append({
            'threshold': round(float(t), 2),
            'precision': precision_score(y_true, pred, zero_division=0),
            'recall': recall_score(y_true, pred, zero_division=0),
            'f1': f1_score(y_true, pred, zero_division=0),
            'missed (FN)': int(((pred == 0) & (y_true == 1)).sum()),
            'false alarms (FP)': int(((pred == 1) & (y_true == 0)).sum()),
        })
    return pd.DataFrame(rows).set_index('threshold')

probs_demo = model_demo.predict_proba(X_test_d)[:, 1]
threshold_table(y_test_d, probs_demo)

### Your Turn — Add costs

Suppose a missed diabetes case (FN) is 5x worse than a false alarm (FP), because a follow-up test is cheap compared with an undiagnosed patient.

In [ ]:
probs_yours = model_yours.predict_proba(X_test_y)[:, 1]
table_y = threshold_table(y_test_y, probs_yours)

cost_fn = 5   # cost of missing a real case
cost_fp = 1   # cost of a false alarm
table_y['total_cost'] = table_y['missed (FN)'] * cost_fn + table_y['false alarms (FP)'] * cost_fp
display(table_y)

best_t = table_y['total_cost'].idxmin()
print(f'Lowest total cost at threshold = {best_t}')

**Your answers:**

1. What threshold minimized total cost, and what happened to recall and precision at that threshold compared with 0.5?
2. Change `cost_fn` to 1 and re-run. Where does the best threshold move, and what does that tell you about how costs drive the decision?

---
## Step 4 — Overfitting and Tuning

Overfitting: a model memorizes the training data instead of learning a pattern that generalizes. The warning sign is a gap between training score and score on unseen data. We use cross-validation (5-fold, on training data only) to estimate performance on unseen data without touching the test set.

In [ ]:
depths = range(1, 16)
train_f1, cv_f1 = [], []
for d in depths:
    tree = DecisionTreeClassifier(max_depth=d, random_state=42)
    tree.fit(X_train_d, y_train_d)
    train_f1.append(f1_score(y_train_d, tree.predict(X_train_d)))
    cv_f1.append(cross_val_score(tree, X_train_d, y_train_d, cv=5, scoring='f1').mean())

plt.figure(figsize=(7, 4))
plt.plot(depths, train_f1, marker='o', label='Training F1')
plt.plot(depths, cv_f1, marker='o', label='Cross-validated F1')
plt.xlabel('Tree depth'); plt.ylabel('F1 score')
plt.title('Overfitting curve — Demo'); plt.legend(); plt.show()

As depth grows, training F1 climbs toward 1.0 while cross-validated F1 flattens or drops. The best depth is where the cross-validated line peaks, not where training is highest.

**Tuning** means searching over settings (hyperparameters) like `max_depth` to find the best combination. The rule: tune using cross-validation on training data, then touch the test set once at the end.

In [ ]:
param_grid = {'max_depth': [2, 3, 4, 5, 6, 8, 10, None], 'min_samples_leaf': [1, 5, 10, 20]}
grid_d = GridSearchCV(DecisionTreeClassifier(random_state=42), param_grid, cv=5, scoring='f1')
grid_d.fit(X_train_d, y_train_d)
tree_tuned_d = grid_d.best_estimator_

def evaluate_model(model, X_tr, y_tr, X_te, y_te):
    pred = model.predict(X_te)
    train_f1 = f1_score(y_tr, model.predict(X_tr))
    test_f1 = f1_score(y_te, pred)
    return {'Accuracy': accuracy_score(y_te, pred), 'Precision': precision_score(y_te, pred, zero_division=0),
            'Recall': recall_score(y_te, pred, zero_division=0), 'F1': test_f1,
            'Train F1': train_f1, 'Overfit gap': train_f1 - test_f1}

tree_full_d = DecisionTreeClassifier(random_state=42).fit(X_train_d, y_train_d)
print('Best settings:', grid_d.best_params_)
pd.DataFrame({'Untuned tree': evaluate_model(tree_full_d, X_train_d, y_train_d, X_test_d, y_test_d),
              'Tuned tree': evaluate_model(tree_tuned_d, X_train_d, y_train_d, X_test_d, y_test_d)}).T

### Your Turn — Overfitting and tuning on your data

In [ ]:
depths = range(1, 16)
train_f1_y, cv_f1_y = [], []
for d in depths:
    tree = DecisionTreeClassifier(max_depth=d, random_state=42)
    tree.fit(X_train_y, y_train_y)
    train_f1_y.append(f1_score(y_train_y, tree.predict(X_train_y)))
    cv_f1_y.append(cross_val_score(tree, X_train_y, y_train_y, cv=5, scoring='f1').mean())

plt.figure(figsize=(7, 4))
plt.plot(depths, train_f1_y, marker='o', label='Training F1')
plt.plot(depths, cv_f1_y, marker='o', label='Cross-validated F1')
plt.xlabel('Tree depth'); plt.ylabel('F1 score')
plt.title('Overfitting curve — Your data'); plt.legend(); plt.show()

In [ ]:
grid_y = GridSearchCV(DecisionTreeClassifier(random_state=42), param_grid, cv=5, scoring='f1')
grid_y.fit(X_train_y, y_train_y)
tree_tuned_y = grid_y.best_estimator_
tree_full_y = DecisionTreeClassifier(random_state=42).fit(X_train_y, y_train_y)

print('Best settings:', grid_y.best_params_)
pd.DataFrame({'Untuned tree': evaluate_model(tree_full_y, X_train_y, y_train_y, X_test_y, y_test_y),
              'Tuned tree': evaluate_model(tree_tuned_y, X_train_y, y_train_y, X_test_y, y_test_y)}).T

**Your answers:**

1. How big is the training-vs-test gap for the unrestricted tree? What settings did tuning pick, and did it shrink the gap?
2. Why is it important that tuning used cross-validation on training data rather than picking settings by looking at test-set scores?

---
## Step 5 — Comparing Models Fairly

There's rarely one "right" model. Standard practice is to train several, compare them on the same test set with the same metrics, and check the overfit gap, not just the test score.

In [ ]:
def compare_models(models, X_tr, y_tr, X_te, y_te):
    return pd.DataFrame({name: evaluate_model(m, X_tr, y_tr, X_te, y_te) for name, m in models.items()}).T

forest_d = RandomForestClassifier(n_estimators=200, random_state=42).fit(X_train_d, y_train_d)
models_demo = {'Logistic Regression': model_demo, 'Decision Tree (tuned)': tree_tuned_d, 'Random Forest': forest_d}
results_demo = compare_models(models_demo, X_train_d, y_train_d, X_test_d, y_test_d)
results_demo[['Precision', 'Recall', 'F1']].plot(kind='bar', figsize=(7, 4), rot=0, ylim=(0, 1.05))
plt.title('Model comparison — Demo'); plt.show()
results_demo

### Your Turn

In [ ]:
forest_y = RandomForestClassifier(n_estimators=200, random_state=42).fit(X_train_y, y_train_y)
models_yours = {'Logistic Regression': model_yours, 'Decision Tree (tuned)': tree_tuned_y, 'Random Forest': forest_y}
results_yours = compare_models(models_yours, X_train_y, y_train_y, X_test_y, y_test_y)
results_yours[['Precision', 'Recall', 'F1']].plot(kind='bar', figsize=(7, 4), rot=0, ylim=(0, 1.05))
plt.title('Model comparison — Your data'); plt.show()
results_yours

**Your answers:**

1. Which model has the best recall, precision, and F1? Is it the same model each time?
2. Which model shows the biggest overfit gap? Note: random forests can show a large gap (near-perfect training score) even when they generalize reasonably — judge by test score first, and treat the gap as a warning sign, not a final verdict.

---
## Step 6 — In-Class Activity (IC-5): Should We Use AI?

A good evaluation ends in a decision, and sometimes the right decision is not to use ML at all. The clinic wants to flag patients for a follow-up test. The alternative they already know: a simple rule, flag anyone with glucose reading 120 or higher. No ML required.

In [ ]:
rule_preds = (X_test_y['Glucose'] >= 120).astype(int)
rule_vs_model = pd.DataFrame({
    'Simple rule (Glucose >= 120)': {'Accuracy': accuracy_score(y_test_y, rule_preds), 'Precision': precision_score(y_test_y, rule_preds),
                                     'Recall': recall_score(y_test_y, rule_preds), 'F1': f1_score(y_test_y, rule_preds)},
    'Logistic Regression': results_yours.loc['Logistic Regression', ['Accuracy', 'Precision', 'Recall', 'F1']],
    'Random Forest': results_yours.loc['Random Forest', ['Accuracy', 'Precision', 'Recall', 'F1']],
}).T
rule_vs_model

Discuss with your group, then write your answers.

1. Baseline: does the ML model clearly beat the simple rule (and the do-nothing baseline from Step 1)? Is the gain worth the added complexity?
2. Cost of errors: what does a missed case cost versus a false alarm here? Does the model's error pattern match what the clinic can tolerate?
3. Human in the loop: should the model make the final call, or only prioritize patients for a nurse to review?
4. Data and fairness: this data covers one specific group, and some measurements are recorded as 0. Would you trust this model on the clinic's actual patients?
5. Monitoring: what would you track after launch to know if the model is going stale?

**Your recommendation** (pick one and justify with specific numbers): deploy the model as-is, deploy with human review, use the simple rule, or don't deploy yet.

**Your answers:**

**To submit for the in-class activity:** upload a screenshot of your rule-vs-model table above along with your written recommendation to the IC-5 module.

---
# Assignment 5 — Evaluate, Tune, and Defend Your Model

Use the same diabetes data and helper functions from above (run all earlier cells first). Write your own thoughts — answers that just restate numbers without explaining why won't get full credit.

---
## Task 1 — Add KNN and tune one model

KNN classifies a patient by looking at the k most similar patients in the training data. Since it works on distances, we scale features first.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

knn_y = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))
knn_y.fit(X_train_y, y_train_y)
models_yours['KNN (k=5)'] = knn_y

results_all = compare_models(models_yours, X_train_y, y_train_y, X_test_y, y_test_y)
results_all[['Precision', 'Recall', 'F1']].plot(kind='bar', figsize=(7, 4), rot=0, ylim=(0, 1.05))
plt.title('All models — Diabetes data'); plt.show()
results_all

In [ ]:
# TODO: choose ONE model (Random Forest or KNN) and tune it with GridSearchCV (training data only)
model_to_tune = RandomForestClassifier(random_state=42)
param_grid_task1 = {'n_estimators': [50, 100, 200], 'max_depth': [3, 5, 8, None]}   # TODO: edit

# --- or, to tune KNN instead ---
# model_to_tune = make_pipeline(StandardScaler(), KNeighborsClassifier())
# param_grid_task1 = {'kneighborsclassifier__n_neighbors': [3, 5, 7, 11, 15, 21]}

grid_task1 = GridSearchCV(model_to_tune, param_grid_task1, cv=5, scoring='f1')   # TODO: try scoring='recall'
grid_task1.fit(X_train_y, y_train_y)
tuned = grid_task1.best_estimator_
untuned = model_to_tune.fit(X_train_y, y_train_y)

print('Best settings:', grid_task1.best_params_)
pd.DataFrame({'Untuned': evaluate_model(untuned, X_train_y, y_train_y, X_test_y, y_test_y),
              'Tuned': evaluate_model(tuned, X_train_y, y_train_y, X_test_y, y_test_y)}).T

**Task 1 answer.** Which model would you pick if the priority is catching as many diabetes cases as possible, and which if the priority is avoiding false alarms? Which scoring metric did you tune for, and why does it fit the clinic's situation? Did tuning help on the test set?

**Your answer:**

## Task 2 — Choose a threshold using costs

Take your best model from Task 1 and pick a decision threshold that minimizes total cost.

In [ ]:
best_model = tuned   # TODO: put your chosen fitted model here

probs_best = best_model.predict_proba(X_test_y)[:, 1]
table_task2 = threshold_table(y_test_y, probs_best)

cost_fn = 5   # TODO: choose your own cost ratio and be ready to justify it
cost_fp = 1
table_task2['total_cost'] = table_task2['missed (FN)'] * cost_fn + table_task2['false alarms (FP)'] * cost_fp
display(table_task2)

best_t2 = table_task2['total_cost'].idxmin()
print(f'Lowest-cost threshold: {best_t2}')

**Task 2 answer.** What cost ratio did you choose and why is it realistic for this clinic? Compared with the default 0.5 threshold, how many more cases did you catch and how many more false alarms did you accept? We picked the threshold by looking at the test set — explain why that's a problem in a real project and what you'd do instead.

**Your answer:**

## Task 3 — Final reflection

In 5-8 sentences: what is your final recommendation for the clinic (model, threshold, human involvement), supported with specific metrics? Give two reasons the test-set results here might be more optimistic than what the clinic would see in real life. What is one thing you would do next to improve or better validate the model?

**Your answer:**